# COVID RNA-seq Pipeline (SRR12544419)

## SRA

In [ ]:
!wget -q https://ftp-trace.ncbi.nlm.nih.gov/sra/sdk/3.2.1/sratoolkit.3.2.1-ubuntu64.tar.gz
!tar -xzf sratoolkit.3.2.1-ubuntu64.tar.gz

In [ ]:
!./sratoolkit.3.2.1-ubuntu64/bin/prefetch SRR12544419

In [ ]:
!./sratoolkit.3.2.1-ubuntu64/bin/fasterq-dump \
    SRR12544419 \
    --split-files \
    -O raw_data

In [ ]:
!ls -lh raw_data

## FASTQ

In [ ]:
!head -n 4 raw_data/SRR12544419_1.fastq

## FastQC
تثبيت ← مجلد ← تشغيل ← html

In [ ]:
!sudo apt-get update -qq
!sudo apt-get install -y fastqc

In [ ]:
!mkdir -p fastqc_reports/raw

In [ ]:
!fastqc \
    raw_data/SRR12544419_1.fastq \
    raw_data/SRR12544419_2.fastq \
    -o fastqc_reports/raw

## Trimming

In [ ]:
!sudo apt-get update -qq
!sudo apt-get install -y trimmomatic

In [ ]:
!TrimmomaticPE -version

In [ ]:
!mkdir -p trimmed

In [ ]:
!TrimmomaticPE -threads 2 \
    raw_data/SRR12544419_1.fastq \
    raw_data/SRR12544419_2.fastq \
    trimmed/SRR12544419_1_paired.fastq \
    trimmed/SRR12544419_1_unpaired.fastq \
    trimmed/SRR12544419_2_paired.fastq \
    trimmed/SRR12544419_2_unpaired.fastq \
    ILLUMINACLIP:/usr/share/trimmomatic/TruSeq3-PE.fa:2:30:10 \
    LEADING:3 \
    TRAILING:3 \
    SLIDINGWINDOW:4:20 \
    MINLEN:36

## Alignment

In [ ]:
!sudo apt-get update -qq
!sudo apt-get install -y hisat2 samtools

In [ ]:
!mkdir -p reference

In [ ]:
!wget -q https://ftp.ensembl.org/pub/release-109/fasta/homo_sapiens/dna/Homo_sapiens.GRCh38.dna.primary_assembly.fa.gz

In [ ]:
!wget -q https://ftp.ensembl.org/pub/release-109/gtf/homo_sapiens/Homo_sapiens.GRCh38.109.gtf.gz

In [ ]:
!gunzip reference/*.gz

In [ ]:
!hisat2-build -p 2 \
reference/Homo_sapiens.GRCh38.dna.primary_assembly.fa \
reference/hg38_index

In [ ]:
!mkdir -p /content/aligned
!hisat2 \
  -x /content/reference/hg38_index \
  -1 /content/trimmed/SRR12544419_1_paired.fastq \
  -2 /content/trimmed/SRR12544419_2_paired.fastq \
  -S /content/aligned/SRR12544419.sam \
  --summary-file /content/aligned/SRR12544419_summary.txt \
  -p 2

In [ ]:
!samtools view -@ 2 -b /content/aligned/SRR12544419.sam > /content/aligned/SRR12544419.bam

In [ ]:
!samtools sort \
  -@ 2 \
  -o /content/aligned/SRR12544419_sorted.bam \
  /content/aligned/SRR12544419.bam

## featureCounts

In [ ]:
!sudo apt-get update -qq
!sudo apt-get install -y subread -qq

In [ ]:
!mkdir -p /content/counts
!featureCounts \
  -T 2 \
  -p \
  -t exon \
  -g gene_id \
  -a /content/reference/Homo_sapiens.GRCh38.109.gtf \
  -o /content/counts/SRR12544419_counts.txt \
  /content/aligned/SRR12544419.bam